In [10]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob
import os
import sys
from torchmetrics.functional import structural_similarity_index_measure as ssim
from run_trained_BM3DUnet_models_finalized import run_ResUnet_4CBCTDen, run_BM3DResUnet_4CBCTDen, run_SwinIR_4CBCTDen, run_SwinIR_4CBCTDen_bm3d, run_BM3DSwinIR_4CBCTDen, run_HARUnet_4CBCTDen, run_HARUnet_4CBCTDen_bm3d, run_BM3DHARUnet_4CBCTDen
import torch.nn.functional as F
from Full_Unet import ConvBlock, Heavy_UNet
#!pip install timm
#!pip install einops
from Uformer_model import Uformer
from network_swinir import SwinIR
from HARUnet_model_v2_1 import HARU_net
from utils import padcrop_resize, Patch_Positions, psnr
from SegUtils import Detect_roi, rescale_cbct_slice, pad_to_patchsize, draw_bounding_box, patching_the_boxes
import imageio.v3 as iio
#from Light_Unet import UNet
#from Full_Unet import ConvBlock, Heavy_UNet
import random
import math
import time
import json  # or use CSV if preferred

from bm3d import bm3d

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
nS, H, W = (512,512,512) 
patch_size = 256

image_size = (512,512)
patch_start_positions = Patch_Positions(image_size, patch_size=256, max_overlap=0)
print(patch_start_positions)

[(0, 0), (256, 0), (0, 256), (256, 256)]


In [ ]:

model_dir = r"C:\Users\au711969\OneDrive - Aarhus universitet\Dentistry_Stuff\My Research projects\CBCT Denoising Project\BM3D_on_CBCT\Codes"

ResUnet_modelname = r"ResUnet_trainedon_noisytorawCBCTs_CadavarData_at_55epochs_newData_latest.pth"
#ResUnet_modelname = r"SupervisedLearning_FullUnet_trainedon_noisytorawCBCTs_CadavarData_at_23epochs_newData.pth"
model_ResUnet = torch.load(os.path.join(model_dir,ResUnet_modelname)).to(device)

#HARUnet_modelname = r"HARUnetv1_trainedon_noisytorawCBCTs_CadavarData_at_20epochs.pth"
HARUnet_modelname = r"HARUnetv2_11_trainedon_noisytorawCBCTs_CadavarData_at_42epochs_.pth"
model_HARUnet = torch.load(os.path.join(model_dir,HARUnet_modelname)).to(device)

HAT_modelname = r"HAT_trainedon_CadavarData_epoch_nr41.pth"
model_HAT = torch.load(os.path.join(model_dir,HAT_modelname)).to(device)

SwinIR_modelname = r"SWINIR_trainedon_noisy2raw_CadavarData_at_39epochs_.pth"
model_SwinIR = torch.load(os.path.join(model_dir,SwinIR_modelname)).to(device)

# 2. Load the state dictionary
Uformer_modelname = r"Uformer_trainedon_noisy2raw_CadavarData_at_40epochs_.pth"
model_Uformer = torch.load(os.path.join(model_dir,Uformer_modelname)).to(device)

# 2. Load the state dictionary
HARU2ResUnet_modelname = r"DistilledHARUnet_ResUNet_trainedon_CBCT_CadavarData_at_50epochs_.pth"
model_HARU2ResUnet = torch.load(os.path.join(model_dir,HARU2ResUnet_modelname)).to(device)
model_QHARU2ResUnet = torch.load(os.path.join(model_dir,HARU2ResUnet_modelname)).to(device).half().cuda()
#model_QHARU2ResUnet = model_QHARU2ResUnet.half().cuda()
# 2. Load the state dictionary
Pruned_HARU2ResUnet_modelname = r"Pruned_25percent_DistilledHARUnet_ResUNet_trainedon_CBCT_CadavarData_0.pth"
model_PrHARU2ResUnet = torch.load(os.path.join(model_dir,Pruned_HARU2ResUnet_modelname)).to(device)
model_QPrHARU2ResUnet = torch.load(os.path.join(model_dir,Pruned_HARU2ResUnet_modelname)).to(device).half().cuda()
#model_QPrHARU2ResUnet = model_QPrHARU2ResUnet.half().cuda()

model_ResUnet.eval()
model_SwinIR.eval()
model_HARUnet.eval()
model_HAT.eval()
model_Uformer.eval()
model_HARU2ResUnet.eval()
model_QHARU2ResUnet.eval()
model_PrHARU2ResUnet.eval()
model_QPrHARU2ResUnet.eval()

C:\Users\au711969\AppData\Local\Temp\ipykernel_10376\3757174628.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_ResUnet = torch.load(os.path.join(model_dir,ResUnet

ResUNet(
  (ConvBlock1): ConvBlock(
    (block): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): LeakyReLU(negative_slope=0.01, inplace=True)
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): LeakyReLU(negative_slope=0.01, inplace=True)
    )
    (conv11): Conv2d(1, 32, kernel_size=(1, 1), stride=(1, 1))
  )
  (pool1): Conv2d(32, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (ConvBlock2): ConvBlock(
    (block): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): LeakyReLU(negative_slope=0.01, inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): LeakyReLU(negative_slope=0.01, inplace=True)
    )
    (conv11): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1))
  )
  (pool2): Conv2d(64, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
  (ConvBlock3): ConvBlock(
    (block): Sequential(
   

In [ ]:
import torch
import torch.nn as nn
import time
import numpy as np

# ---------------------------------------------
# 1. Create synthetic 3D CBCT volume (512,512,512)
# ---------------------------------------------


# Convert to shape (D, H, W)
nS, H, W = (512,512,512) 
patch_size = 256

image_size = (512,512)
patch_start_positions = Patch_Positions(image_size, patch_size=256, max_overlap=0)


no_runs = 100

All_ResUnet_time_per_scan = np.zeros(no_runs)
All_Uformer_time_per_scan = np.zeros(no_runs)
All_SwinIR_time_per_scan = np.zeros(no_runs)
All_HARUnet_time_per_scan  = np.zeros(no_runs)
All_HAT_time_per_scan  = np.zeros(no_runs)
All_HARU2ResU_time_per_scan  = np.zeros(no_runs)
All_QHARU2ResU_time_per_scan = np.zeros(no_runs)
All_PrHARU2ResU_time_per_scan  = np.zeros(no_runs)
All_QPrHARU2ResU_time_per_scan = np.zeros(no_runs)

for k in range(no_runs):
    volume_size = (512, 512, 512)
    volume = torch.randn(volume_size, dtype=torch.float32)  # random volume

    ###################### ResU-Net ###################
#
    ## Track inference time
    #start_time = time.time()
#
    #volume=volume.to(device) 
    #recon_volume = torch.zeros_like(volume).to(device)
    ## Loop through slices (depth dimension)
    #for z in range(nS):
#
    #    slice_2d = volume[z]  # shape = (512,512)
#
    #    # Reconstruct slice buffer
    #    recon_slice = torch.zeros((H, W), dtype=torch.float32)
#
    #    # Process in 256x256 patches
    #    for (i,j) in patch_start_positions:
#
    #        patch = slice_2d[i:i+patch_size, j:j+patch_size]
    #        # Ensure patch is full-sized (512x512 fits 256 exactly)
    #        if patch.shape != (patch_size, patch_size):
    #            continue
    #        # Model inference
    #        inp = patch.unsqueeze(0).unsqueeze(0) # (1,1,256,256)
    #        with torch.no_grad():
    #            out = model_ResUnet(inp)        # (256,256)
    #        # Place back into reconstructed slice
    #        recon_slice[i:i+patch_size, j:j+patch_size] = out
#
    #    # Assign reconstructed slice back to the 3D volume
    #    recon_volume[z] = recon_slice
#
    #end_time = time.time()
#
    ## ---------------------------------------------
    ## 4. Report timing
    ## ---------------------------------------------
    #ResUnet_time_per_scan = end_time - start_time
    #All_ResUnet_time_per_scan[k] = ResUnet_time_per_scan
#
    #################################### Uformer #################################
    ## Track inference time
    #start_time = time.time()
#
    #volume=volume.to(device) 
    #recon_volume = torch.zeros_like(volume).to(device)
    ## Loop through slices (depth dimension)
    #for z in range(nS):
#
    #    slice_2d = volume[z]  # shape = (512,512)
#
    #    # Reconstruct slice buffer
    #    recon_slice = torch.zeros((H, W), dtype=torch.float32)
#
    #    # Process in 256x256 patches
    #    for (i,j) in patch_start_positions:
#
    #        patch = slice_2d[i:i+patch_size, j:j+patch_size]
    #        # Ensure patch is full-sized (512x512 fits 256 exactly)
    #        if patch.shape != (patch_size, patch_size):
    #            continue
    #        # Model inference
    #        inp = patch.unsqueeze(0).unsqueeze(0) # (1,1,256,256)
    #        with torch.no_grad():
    #            out = model_Uformer(inp)        # (256,256)
    #        # Place back into reconstructed slice
    #        recon_slice[i:i+patch_size, j:j+patch_size] = out
#
    #    # Assign reconstructed slice back to the 3D volume
    #    recon_volume[z] = recon_slice
#
    #end_time = time.time()
#
    ## ---------------------------------------------
    ## 4. Report timing
    ## ---------------------------------------------
    #Uformer_time_per_scan = end_time - start_time
    #All_Uformer_time_per_scan[k] = Uformer_time_per_scan
#
    #################################### SwinIR #################################
    ## Track inference time
    #start_time = time.time()
#
    #volume=volume.to(device) 
    #recon_volume = torch.zeros_like(volume).to(device)
    ## Loop through slices (depth dimension)
    #for z in range(nS):
#
    #    slice_2d = volume[z]  # shape = (512,512)
#
    #    # Reconstruct slice buffer
    #    recon_slice = torch.zeros((H, W), dtype=torch.float32)
#
    #    # Process in 256x256 patches
    #    for (i,j) in patch_start_positions:
#
    #        patch = slice_2d[i:i+patch_size, j:j+patch_size]
    #        # Ensure patch is full-sized (512x512 fits 256 exactly)
    #        if patch.shape != (patch_size, patch_size):
    #            continue
    #        # Model inference
    #        inp = patch.unsqueeze(0).unsqueeze(0) # (1,1,256,256)
    #        with torch.no_grad():
    #            out = model_SwinIR(inp)        # (256,256)
    #        # Place back into reconstructed slice
    #        recon_slice[i:i+patch_size, j:j+patch_size] = out
#
    #    # Assign reconstructed slice back to the 3D volume
    #    recon_volume[z] = recon_slice
#
    #end_time = time.time()
#
    ## ---------------------------------------------
    ## 4. Report timing
    ## ---------------------------------------------
    #SwinIR_time_per_scan = end_time - start_time
    #All_SwinIR_time_per_scan[k] = SwinIR_time_per_scan
    #
    #################################### HARUnet #################################
    ## Track inference time
    #start_time = time.time()
#
    #volume=volume.to(device) 
    #recon_volume = torch.zeros_like(volume).to(device)
    ## Loop through slices (depth dimension)
    #for z in range(nS):
#
    #    slice_2d = volume[z]  # shape = (512,512)
#
    #    # Reconstruct slice buffer
    #    recon_slice = torch.zeros((H, W), dtype=torch.float32)
#
    #    # Process in 256x256 patches
    #    for (i,j) in patch_start_positions:
#
    #        patch = slice_2d[i:i+patch_size, j:j+patch_size]
    #        # Ensure patch is full-sized (512x512 fits 256 exactly)
    #        if patch.shape != (patch_size, patch_size):
    #            continue
    #        # Model inference
    #        inp = patch.unsqueeze(0).unsqueeze(0) # (1,1,256,256)
    #        with torch.no_grad():
    #            out = model_HARUnet(inp)        # (256,256)
    #        # Place back into reconstructed slice
    #        recon_slice[i:i+patch_size, j:j+patch_size] = out
#
    #    # Assign reconstructed slice back to the 3D volume
    #    recon_volume[z] = recon_slice
#
    #end_time = time.time()
#
    ## ---------------------------------------------
    ## 4. Report timing
    ## ---------------------------------------------
    #HARUnet_time_per_scan = end_time - start_time
    #All_HARUnet_time_per_scan[k] = HARUnet_time_per_scan
#
    ################################### HARUnet #################################
    # Track inference time
    start_time = time.time()

    volume=volume.to(device) 
    recon_volume = torch.zeros_like(volume).to(device)
    # Loop through slices (depth dimension)
    for z in range(nS):

        slice_2d = volume[z]  # shape = (512,512)

        # Reconstruct slice buffer
        recon_slice = torch.zeros((H, W), dtype=torch.float32)

        # Process in 256x256 patches
        for (i,j) in patch_start_positions:

            patch = slice_2d[i:i+patch_size, j:j+patch_size]
            # Ensure patch is full-sized (512x512 fits 256 exactly)
            if patch.shape != (patch_size, patch_size):
                continue
            # Model inference
            inp = patch.unsqueeze(0).unsqueeze(0) # (1,1,256,256)
            with torch.no_grad():
                out = model_HAT(inp)        # (256,256)
            # Place back into reconstructed slice
            recon_slice[i:i+patch_size, j:j+patch_size] = out

        # Assign reconstructed slice back to the 3D volume
        recon_volume[z] = recon_slice

    end_time = time.time()

    # ---------------------------------------------
    # 4. Report timing
    # ---------------------------------------------
    HAT_time_per_scan = end_time - start_time
    All_HAT_time_per_scan[k] = HAT_time_per_scan

#################################### HARU2ResU #################################
#    # Track inference time
#    start_time = time.time()
#
#    volume=volume.to(device) 
#    recon_volume = torch.zeros_like(volume).to(device)
#    # Loop through slices (depth dimension)
#    for z in range(nS):
#
#        slice_2d = volume[z]  # shape = (512,512)
#
#        # Reconstruct slice buffer
#        recon_slice = torch.zeros((H, W), dtype=torch.float32)
#
#        # Process in 256x256 patches
#        for (i,j) in patch_start_positions:
#
#            patch = slice_2d[i:i+patch_size, j:j+patch_size]
#            # Ensure patch is full-sized (512x512 fits 256 exactly)
#            if patch.shape != (patch_size, patch_size):
#                continue
#            # Model inference
#            inp = patch.unsqueeze(0).unsqueeze(0) # (1,1,256,256)
#            with torch.no_grad():
#                out = model_HARU2ResUnet(inp)        # (256,256)
#            # Place back into reconstructed slice
#            recon_slice[i:i+patch_size, j:j+patch_size] = out
#
#        # Assign reconstructed slice back to the 3D volume
#        recon_volume[z] = recon_slice
#
#    end_time = time.time()
#
#    # ---------------------------------------------
#    # 4. Report timing
#    # ---------------------------------------------
#    HARU2ResU_time_per_scan = end_time - start_time
#    All_HARU2ResU_time_per_scan[k] = HARU2ResU_time_per_scan
#
#################################### QHARU2ResU #################################
#    # Track inference time
#    start_time = time.time()
#
#    volume=volume.to(device) 
#    recon_volume = torch.zeros_like(volume).to(device)
#    # Loop through slices (depth dimension)
#    for z in range(nS):
#
#        slice_2d = volume[z]  # shape = (512,512)
#
#        # Reconstruct slice buffer
#        recon_slice = torch.zeros((H, W), dtype=torch.float32)
#
#        # Process in 256x256 patches
#        for (i,j) in patch_start_positions:
#
#            patch = slice_2d[i:i+patch_size, j:j+patch_size]
#            # Ensure patch is full-sized (512x512 fits 256 exactly)
#            if patch.shape != (patch_size, patch_size):
#                continue
#            # Model inference
#            inp = patch.unsqueeze(0).unsqueeze(0) # (1,1,256,256)
#            with torch.no_grad():
#                out = model_QHARU2ResUnet(inp.half().cuda())        # (256,256)
#            # Place back into reconstructed slice
#            recon_slice[i:i+patch_size, j:j+patch_size] = out
#
#        # Assign reconstructed slice back to the 3D volume
#        recon_volume[z] = recon_slice
#
#    end_time = time.time()
#
#    # ---------------------------------------------
#    # 4. Report timing
#    # ---------------------------------------------
#    QHARU2ResU_time_per_scan = end_time - start_time
#    All_QHARU2ResU_time_per_scan[k] = QHARU2ResU_time_per_scan
#
#################################### PrHARU2ResU #################################
#    # Track inference time
#    start_time = time.time()
#
#    volume=volume.to(device) 
#    recon_volume = torch.zeros_like(volume).to(device)
#    # Loop through slices (depth dimension)
#    for z in range(nS):
#
#        slice_2d = volume[z]  # shape = (512,512)
#
#        # Reconstruct slice buffer
#        recon_slice = torch.zeros((H, W), dtype=torch.float32)
#
#        # Process in 256x256 patches
#        for (i,j) in patch_start_positions:
#
#            patch = slice_2d[i:i+patch_size, j:j+patch_size]
#            # Ensure patch is full-sized (512x512 fits 256 exactly)
#            if patch.shape != (patch_size, patch_size):
#                continue
#            # Model inference
#            inp = patch.unsqueeze(0).unsqueeze(0) # (1,1,256,256)
#            with torch.no_grad():
#                out = model_PrHARU2ResUnet(inp)        # (256,256)
#            # Place back into reconstructed slice
#            recon_slice[i:i+patch_size, j:j+patch_size] = out
#
#        # Assign reconstructed slice back to the 3D volume
#        recon_volume[z] = recon_slice
#
#    end_time = time.time()
#
#    # ---------------------------------------------
#    # 4. Report timing
#    # ---------------------------------------------
#    PrHARU2ResU_time_per_scan = end_time - start_time
#    All_PrHARU2ResU_time_per_scan[k] = PrHARU2ResU_time_per_scan
#
#    ################################### PrHARU2ResU #################################
#    # Track inference time
#    start_time = time.time()
#
#    volume=volume.to(device) 
#    recon_volume = torch.zeros_like(volume).to(device)
#    # Loop through slices (depth dimension)
#    for z in range(nS):
#
#        slice_2d = volume[z]  # shape = (512,512)
#
#        # Reconstruct slice buffer
#        recon_slice = torch.zeros((H, W), dtype=torch.float32)
#
#        # Process in 256x256 patches
#        for (i,j) in patch_start_positions:
#
#            patch = slice_2d[i:i+patch_size, j:j+patch_size]
#            # Ensure patch is full-sized (512x512 fits 256 exactly)
#            if patch.shape != (patch_size, patch_size):
#                continue
#            # Model inference
#            inp = patch.unsqueeze(0).unsqueeze(0) # (1,1,256,256)
#            with torch.no_grad():
#                out = model_QPrHARU2ResUnet(inp.half().cuda())        # (256,256)
#            # Place back into reconstructed slice
#            recon_slice[i:i+patch_size, j:j+patch_size] = out
#
#        # Assign reconstructed slice back to the 3D volume
#        recon_volume[z] = recon_slice
#
#    end_time = time.time()
#
#    # ---------------------------------------------
#    # 4. Report timing
#    # ---------------------------------------------
#    QPrHARU2ResU_time_per_scan = end_time - start_time
#    All_QPrHARU2ResU_time_per_scan[k] = QPrHARU2ResU_time_per_scan
#
#    
#    print(f"\n Uformer inference time: per slice: {Uformer_time_per_scan / nS:.4f} seconds, per 512×512×512 volume: {Uformer_time_per_scan:.3f} seconds")
#    print(f"\n SwinIR inference time: per slice: {SwinIR_time_per_scan / nS:.4f} seconds, per 512×512×512 volume: {SwinIR_time_per_scan:.3f} seconds")
    print(f"\n HARU-Net inference time: per slice: {HAT_time_per_scan / nS:.4f} seconds, per 512×512×512 volume: {HAT_time_per_scan:.3f} seconds")
#    print(f"\n HARU-Net inference time: per slice: {HARUnet_time_per_scan / nS:.4f} seconds, per 512×512×512 volume: {HARUnet_time_per_scan:.3f} seconds")
#    print(f"\n ResU-Net inference time: per slice: {ResUnet_time_per_scan / nS:.4f} seconds, per 512×512×512 volume: {ResUnet_time_per_scan:.3f} seconds")
#    print(f"\n HARU2ResU inference time: per slice: {HARU2ResU_time_per_scan / nS:.4f} seconds, per 512×512×512 volume: {HARU2ResU_time_per_scan:.3f} seconds")
#    print(f"\n QHARU2ResU inference time: per slice: {QHARU2ResU_time_per_scan / nS:.4f} seconds, per 512×512×512 volume: {QHARU2ResU_time_per_scan:.3f} seconds")
#    print(f"\n PrHARU2ResU inference time: per slice: {PrHARU2ResU_time_per_scan / nS:.4f} seconds, per 512×512×512 volume: {PrHARU2ResU_time_per_scan:.3f} seconds")
#    print(f"\n QPrHARU2ResU inference time: per slice: {QPrHARU2ResU_time_per_scan / nS:.4f} seconds, per 512×512×512 volume: {QPrHARU2ResU_time_per_scan:.3f} seconds")
    # If using GPU, also measure avg time per slice
    print(f" ")



# If using GPU, also measure avg time per slice
print(f" ")


 Uformer inference time: per slice: 0.5090 seconds, per 512×512×512 volume: 260.618 seconds

 SwinIR inference time: per slice: 1.0534 seconds, per 512×512×512 volume: 539.345 seconds

 HARU-Net inference time: per slice: 0.2329 seconds, per 512×512×512 volume: 119.230 seconds

 ResU-Net inference time: per slice: 0.0246 seconds, per 512×512×512 volume: 12.597 seconds

 HARU2ResU inference time: per slice: 0.0394 seconds, per 512×512×512 volume: 20.152 seconds

 QHARU2ResU inference time: per slice: 0.0370 seconds, per 512×512×512 volume: 18.964 seconds

 PrHARU2ResU inference time: per slice: 0.0243 seconds, per 512×512×512 volume: 12.429 seconds

 QPrHARU2ResU inference time: per slice: 0.0228 seconds, per 512×512×512 volume: 11.699 seconds
 

 Uformer inference time: per slice: 0.5131 seconds, per 512×512×512 volume: 262.691 seconds

 SwinIR inference time: per slice: 1.0522 seconds, per 512×512×512 volume: 538.726 seconds

 HARU-Net inference time: per slice: 0.2322 seconds, per 5

KeyboardInterrupt: 

In [7]:
All_Uformer_time_per_scan_ = All_Uformer_time_per_scan[All_Uformer_time_per_scan != 0]
print(np.size(All_Uformer_time_per_scan_ ))
All_SwinIR_time_per_scan_ = All_SwinIR_time_per_scan[All_SwinIR_time_per_scan != 0]
print(np.size(All_SwinIR_time_per_scan_ ))
All_HARUnet_time_per_scan_ = All_HARUnet_time_per_scan[All_HARUnet_time_per_scan != 0]
print(np.size(All_HARUnet_time_per_scan_ ))
All_ResUnet_time_per_scan_ = All_ResUnet_time_per_scan[All_ResUnet_time_per_scan != 0]
All_HARU2ResU_time_per_scan_ = All_HARU2ResU_time_per_scan[All_HARU2ResU_time_per_scan != 0]
All_QHARU2ResU_time_per_scan_ = All_QHARU2ResU_time_per_scan[All_QHARU2ResU_time_per_scan != 0]
All_PrHARU2ResU_time_per_scan_ = All_PrHARU2ResU_time_per_scan[All_PrHARU2ResU_time_per_scan != 0]
All_QPrHARU2ResU_time_per_scan_ = All_QPrHARU2ResU_time_per_scan[All_QPrHARU2ResU_time_per_scan != 0]
print(np.size(All_QPrHARU2ResU_time_per_scan_ ))

67
66
66
66


In [8]:

print(f"\n Average Uformer inference time per 512×512×512 volume: {np.mean(All_Uformer_time_per_scan[:66]):.3f} seconds")
print(f"\n Average SwinIR inference time per 512×512×512 volume: {np.mean(All_SwinIR_time_per_scan[:66]):.3f} seconds")
print(f"\n Average HARU-Net inference time per 512×512×512 volume: {np.mean(All_HARUnet_time_per_scan[:66]):.3f} seconds")
print(f"\n Average ResU-Net inference tim per 512×512×512 volume: {np.mean(All_ResUnet_time_per_scan[:66]):.3f} seconds")
print(f"\n Average HARU2ResUnet inference time per 512×512×512 volume: {np.mean(All_HARU2ResU_time_per_scan[:66]):.3f} seconds")
print(f"\n Average QHARU2ResUnet inference time per 512×512×512 volume: {np.mean(All_QHARU2ResU_time_per_scan[:66]):.3f} seconds")
print(f"\n Average PrHARU2ResUnet inference time per 512×512×512 volume: {np.mean(All_PrHARU2ResU_time_per_scan[:66]):.3f} seconds")
print(f"\n Average QPrHARU2ResUnet inference time per 512×512×512 volume: {np.mean(All_QPrHARU2ResU_time_per_scan[:66]):.3f} seconds")


 Average Uformer inference time per 512×512×512 volume: 252.498 seconds

 Average SwinIR inference time per 512×512×512 volume: 517.171 seconds

 Average HARU-Net inference time per 512×512×512 volume: 114.918 seconds

 Average ResU-Net inference tim per 512×512×512 volume: 11.895 seconds

 Average HARU2ResUnet inference time per 512×512×512 volume: 19.406 seconds

 Average QHARU2ResUnet inference time per 512×512×512 volume: 18.814 seconds

 Average PrHARU2ResUnet inference time per 512×512×512 volume: 11.657 seconds

 Average QPrHARU2ResUnet inference time per 512×512×512 volume: 10.699 seconds


In [ ]:
.import numpy as np

# -----------------------------------------
# Per-volume inference times for 10 runs
# -----------------------------------------

bm3d = np.array([
    2309.874, 2347.415, 2355.028, 2318.645, 2319.415,
    2315.818, 2317.410, 2315.994, 2318.714, 2316.304
])

resunet = np.array([
    12.172, 10.585, 11.913, 12.890, 12.844,
    12.908, 12.796, 12.948, 12.727, 11.239
])

uformer = np.array([
    256.514, 240.586, 263.780, 261.851, 262.919,
    264.926, 264.173, 262.935, 264.347, 236.938
])

swinir = np.array([
    531.914, 489.293, 522.876, 555.200, 554.998,
    554.963, 553.846, 555.100, 504.454, 488.716
])

harunet = np.array([
    129.277, 122.413, 120.364, 118.182, 118.922,
    120.204, 119.045, 118.915, 113.349, 110.604
])

# -----------------------------------------
# Compute averages
# -----------------------------------------
avg_bm3d   = bm3d.mean()
avg_resu   = resunet.mean()
avg_uformer= uformer.mean()
avg_swinir = swinir.mean()
avg_haru   = harunet.mean()

# -----------------------------------------
# Print averages
# -----------------------------------------
print("Average inference times (per 512×512×512 scan):")
print(f"BM3D     : {avg_bm3d:.3f} s")
print(f"ResU-Net : {avg_resu:.3f} s")
print(f"Uformer  : {avg_uformer:.3f} s")
print(f"SwinIR   : {avg_swinir:.3f} s")
print(f"HARU-Net : {avg_haru:.3f} s")

# -----------------------------------------
# LaTeX Table
# -----------------------------------------

print("\n\nLaTeX Table:")
print(r"""
\begin{table}[h!]
\centering
\caption{Average inference time per 512$\times$512$\times$512 CBCT scan over 10 runs.}
\begin{tabular}{|c|c|}
\hline
\textbf{Method} & \textbf{Avg. Time (s)} \\ \hline
BM3D & %.3f \\ \hline
ResU-Net & %.3f \\ \hline
Uformer & %.3f \\ \hline
SwinIR & %.3f \\ \hline
HARU-Net & %.3f \\ \hline
\end{tabular}
\end{table}
""" % (avg_bm3d, avg_resu, avg_uformer, avg_swinir, avg_haru))


SyntaxError: invalid syntax (3010451180.py, line 1)